# CubicasA5K — YOLOv8 Floor Plan Detection
**Classes:** door · wall · window  
**Dataset:** 4178 train / 400 val / 400 test  

### Setup instructions before running:
1. Upload your dataset zip to Google Drive (see Cell 3 for folder structure)
2. Runtime → Change runtime type → **T4 GPU**
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
%pip install -q ultralytics
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
#
# Upload your dataset to Google Drive with this structure:
#
#   MyDrive/
#     cubicasa5k/
#       cubicasa5k-2-6/   ← contains data.yaml
#       train/
#         images/
#         labels/
#       valid/
#         images/
#         labels/
#       test/
#         images/
#         labels/
#
# TIP: zip the cubicasa5k folder and upload the zip, then unzip here.

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 4: Unzip dataset (run only if you uploaded a zip) ────────────────────
import os

ZIP_PATH = '/content/drive/MyDrive/cubicasa5k.zip'   # adjust if needed
EXTRACT_TO = '/content/datasets'

os.makedirs(EXTRACT_TO, exist_ok=True)

if os.path.exists(ZIP_PATH):
    print('Unzipping dataset...')
    !unzip -q "{ZIP_PATH}" -d "{EXTRACT_TO}"
    print('Done.')
else:
    print(f'No zip found at {ZIP_PATH}')
    print('If you already extracted, skip this cell.')

!ls /content/datasets/

In [ ]:
# ── Cell 5: Rewrite data.yaml with absolute Colab paths ───────────────────────
import yaml, os

# After unzipping cubicasa5k.zip to /content/datasets,
# the structure is: /content/datasets/cubicasa5k-2-6/train/images etc.
DATASET_ROOT = '/content/datasets/cubicasa5k-2-6'
colab_yaml_path = '/content/data.yaml'

data = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 3,
    'names': ['door', 'wall', 'window']
}

with open(colab_yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('data.yaml written:')
!cat /content/data.yaml

# Verify images exist
train_count = len(os.listdir(os.path.join(DATASET_ROOT, 'train', 'images')))
val_count   = len(os.listdir(os.path.join(DATASET_ROOT, 'valid', 'images')))
print(f'\nTrain images: {train_count}')
print(f'Val images:   {val_count}')

In [ ]:
# ── Cell 6: Auto-select batch size based on GPU VRAM ─────────────────────────
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {vram_gb:.1f} GB')

if vram_gb >= 38:    # A100
    BATCH = 32
    MODEL = 'yolov8m.pt'
    IMGSZ = 1024
elif vram_gb >= 15:  # T4 / V100
    BATCH = 16
    MODEL = 'yolov8m.pt'   # upgraded from yolov8s for better wall recall
    IMGSZ = 1024
else:                # smaller GPU
    BATCH = 8
    MODEL = 'yolov8s.pt'
    IMGSZ = 640

print(f'Selected model : {MODEL}')
print(f'Selected batch : {BATCH}')
print(f'Selected imgsz : {IMGSZ}')

In [ ]:
# ── Cell 7: TRAIN ─────────────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data        = '/content/data.yaml',
    epochs      = 80,
    imgsz       = IMGSZ,        # 1024 on T4/A100 — key for thin wall detection
    batch       = BATCH,
    device      = 0,
    workers     = 4,
    cache       = True,
    patience    = 20,
    multi_scale = True,         # handles walls of varying lengths
    cos_lr      = True,
    close_mosaic= 15,
    optimizer   = 'AdamW',
    lr0         = 0.001,        # lower lr for larger yolov8m model
    lrf         = 0.01,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
    weight_decay     = 0.0005,
    box         = 7.5,
    cls         = 0.5,
    dfl         = 1.5,
    iou         = 0.7,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.15,         # slightly more copy-paste for wall instances
    degrees     = 10.0,         # more rotation — walls appear at all angles
    translate   = 0.1,
    scale       = 0.6,          # more scale variation for varying wall lengths
    shear       = 2.0,
    fliplr      = 0.5,
    flipud      = 0.25,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    perspective = 0.0001,
    project     = '/content/runs',
    name        = 'cubicasa_yolo',
    exist_ok    = True,
)

In [ ]:
# ── Cell 8: Validate best model ───────────────────────────────────────────────
from ultralytics import YOLO

best_model = YOLO('/content/runs/cubicasa_yolo/weights/best.pt')

metrics = best_model.val(
    data    = '/content/data.yaml',
    imgsz   = IMGSZ,
    split   = 'val',
    iou     = 0.45,   # lower NMS threshold — walls heavily overlap at junctions
    conf    = 0.001,
    device  = 0,
)

print('\n========== RESULTS ==========')
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print('==============================')
for i, name in enumerate(['door', 'wall', 'window']):
    p  = metrics.box.p[i]
    r  = metrics.box.r[i]
    m  = metrics.box.ap50[i]
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f"  {name:6s}  P={p:.3f}  R={r:.3f}  mAP50={m:.3f}  F1={f1:.3f}")

In [ ]:
# ── Cell 9: Show training curves ──────────────────────────────────────────────
from IPython.display import Image, display
import os

run_dir = '/content/runs/cubicasa_yolo'

for img in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    path = os.path.join(run_dir, img)
    if os.path.exists(path):
        print(f'--- {img} ---')
        display(Image(path))

In [ ]:
# ── Cell 10: Save best model to Google Drive ───────────────────────────────────
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/cubicasa5k/trained_models'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy(
    '/content/runs/cubicasa_yolo/weights/best.pt',
    os.path.join(SAVE_DIR, 'best.pt')
)

shutil.copy(
    '/content/runs/cubicasa_yolo/weights/last.pt',
    os.path.join(SAVE_DIR, 'last.pt')
)

# Also save the full results folder
shutil.copytree(
    '/content/runs/cubicasa_yolo',
    os.path.join(SAVE_DIR, 'run_results'),
    dirs_exist_ok=True
)

print(f'Saved to Google Drive: {SAVE_DIR}')
print(os.listdir(SAVE_DIR))